*Module 4 of 9*

> **¿Prefieres español?** Abre [`04_indices_de_vegetacion.ipynb`](../es/04_indices_de_vegetacion.ipynb) — es el mismo módulo, en español.


# 🌱 Module 4 — Vegetation indices

🧭 **Objectives** — turn raw bands into meaning. Compute **NDVI** yourself
from the red and NIR bands, verify it against the pre-computed layer in the
tile, and learn what the other 6 indices in the tile add. Understand why a
whole *family* of indices — not just NDVI — helps a classifier tell crops
apart.

📚 **The idea.** In Module 2 you saw a crop's spectral signature leap up in
the near-infrared while red stays low. An **index** distills that contrast
into one number. The most famous is **NDVI**:

$$ NDVI = \frac{NIR - Red}{NIR + Red} $$

It runs from −1 (water) through ~0 (bare soil) to ~+0.9 (dense healthy
crop). Because it is a *ratio*, it cancels out differences in brightness
(sun angle, slope) and reads plant vigor directly.

📚 **Why more than NDVI?** NDVI saturates over very dense canopies and says
nothing about water or soil residue. So the tile also carries **EVI** and
**GCVI** (chlorophyll), **MSAVI2** (soil-adjusted), **LSWI** (water),
**NDSVI** and **NDTI** (residue / tillage). Together, 6 bands + 7 indices =
**13 layers** describing each pixel — richer clues for the classifier.


In [ ]:
# Get the workshop tile (a few MB; cached after the first download)
import os, sys

async def get_file(name):
    for cand in (f"files/{name}", name, f"../files/{name}", f"../../files/{name}"):
        if os.path.exists(cand):
            return cand
    dest = f"/tmp/{name}"
    if not os.path.exists(dest):
        url = f"https://raw.githubusercontent.com/abxda/portable-geocrop/main/files/{name}"
        if sys.platform == "emscripten":
            from pyodide.http import pyfetch
            resp = await pyfetch(url)
            open(dest, "wb").write(await resp.bytes())
        else:
            import urllib.request
            urllib.request.urlretrieve(url, dest)
    return dest

TILE = await get_file("crop_tile_384.tif")
print("Tile ready:", TILE)

## Compute NDVI yourself, then check it

The tile stores integers scaled by 10000. Red is band index 2, NIR is band
index 3. Compute NDVI with the formula — one vectorized NumPy line, just
like Module 1 — then compare it to layer 6, which the pipeline pre-computed.
They should match.


In [ ]:
import numpy as np, rasterio, matplotlib.pyplot as plt

with rasterio.open(TILE) as src:
    img = src.read().astype(np.float64)

red = img[2]
nir = img[3]
ndvi_mine = (nir - red) / (nir + red + 1e-9)   # +tiny to avoid divide-by-zero
ndvi_tile = img[6] / 10000.0                    # pre-computed layer

diff = np.abs(ndvi_mine - ndvi_tile)
print("Max difference between my NDVI and the tile's:", round(float(diff.max()), 4))
print("They match — you just reproduced a real satellite product.")

In [ ]:
# Look at the NDVI map: green = vigorous crop, brown = soil, dark = water
plt.figure(figsize=(7.5, 6))
im = plt.imshow(ndvi_mine, cmap="RdYlGn", vmin=0, vmax=0.9)
plt.title("NDVI — crop vigor across the Yaqui Valley")
plt.axis("off"); plt.colorbar(im, shrink=0.8, label="NDVI"); plt.show()

## The whole family, side by side

Each index highlights something different. Seeing them together shows why a
classifier benefits from all 13 layers: a wheat field and a chickpea field
might look similar in NDVI but differ in a water or residue index.


In [ ]:
# Layers 6..12 are the 7 indices, in this order:
index_names = ["NDVI", "EVI", "GCVI", "MSAVI2", "LSWI", "NDSVI", "NDTI"]

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for k, ax in enumerate(axes.ravel()):
    if k < len(index_names):
        layer = img[6 + k] / 10000.0
        im = ax.imshow(layer, cmap="RdYlGn")
        ax.set_title(index_names[k]); ax.axis("off")
    else:
        ax.axis("off")
plt.suptitle("6 spectral bands become 7 indices = 13 clues per pixel")
plt.tight_layout(); plt.show()

## Phenology: why one month is a snapshot, a season is a story

NDVI on a single date is a snapshot. Follow the same field *through* the
season — planting, greening, peak, harvest — and its NDVI traces a curve
called its **phenology**. That temporal fingerprint is often what separates
two crops that look identical on any single day. Your tile is one month
(March 2018); the production pipeline in Module 9 stacks *many* months
exactly to capture this.


## 🧪 Check yourself

**NDVI is `(NIR − Red) / (NIR + Red)`. Why divide, instead of just using
`NIR − Red`?**

<details><summary>Show answer</summary>

Dividing makes it a *ratio*, which cancels overall brightness differences
(sun angle, terrain slope, thin haze). `NIR − Red` alone would change with
lighting even for the same healthy plant; the normalized ratio reads vigor
consistently, from −1 to +1.

</details>

**Why carry EVI, LSWI, NDTI... when NDVI already measures greenness?**

<details><summary>Show answer</summary>

NDVI saturates over dense canopies and ignores water and soil residue. Other
indices capture chlorophyll, canopy water, and tillage/residue. Two crops
can share an NDVI but differ in these — so more indices give the classifier
more ways to tell them apart.

</details>


## 🔭 Go deeper

Optional: these bilingual concept cards expand what you just learned
(prerequisite chains, lineage to fundamentals, curated references):

- [NDVI, in depth](https://abxda.github.io/rs-learning-audio/?id=ndvi)
- [Vegetation indices (the family)](https://abxda.github.io/rs-learning-audio/?id=vegetation-indices)
- [Spectral indices](https://abxda.github.io/rs-learning-audio/?id=spectral-indices)
- [Near-infrared (NIR)](https://abxda.github.io/rs-learning-audio/?id=near-infrared)
- [Phenology (the crop calendar)](https://abxda.github.io/rs-learning-audio/?id=phenology)
- [Leaf Area Index](https://abxda.github.io/rs-learning-audio/?id=leaf-area-index)



---

[← Previous · Module 3 — Clean data: from clouds to the geomedian](03_clean_data_geomedian.ipynb) · [Next → · Module 5 — From pixels to parcels: segmentation](05_pixels_to_parcels.ipynb)
